# Train mmBERT mask-targeted classifier — StereoQueerEval Task A

Binary stereotype detection (`yes` / `no`). Works on **Kaggle** and Colab.
Clones `src/`, finds the TSVs automatically, trains both readout variants, and
plots losses, macro-F1 and confusion matrices.

In [ ]:
import os
import subprocess, sys

if not os.path.isdir('Try'):
    subprocess.run(f'git clone --depth 1 https://github.com/AmnO-O/Try.git', shell=True, check=True)
else:
    subprocess.run('git -C Try pull --ff-only', shell=True, check=True)
os.chdir('Try')
os.makedirs('data', exist_ok=True)
subprocess.run('pip install -q -r requirements.txt', shell=True, check=True)
print('kernel :', 'Kaggle' if 'KAGGLE_KERNEL_RUN_TYPE' in os.environ else 'Colab' if 'COLAB_GPU' in os.environ else 'local')
print('repo   :', os.getcwd())
!ls src

## 1. Locate the TSVs

Downloads are gitignored, so the clone has no data. This cell searches the usual
places — your Kaggle dataset path, Colab `/content`, and a local `data/` — and
sets `LGBT_DATA_DIR` to the directory that actually contains the files.

In [ ]:
import glob, os

candidates = []
if 'LGBT_DATA_DIR' in os.environ and os.path.isdir(os.environ['LGBT_DATA_DIR']):
    candidates.append(os.environ['LGBT_DATA_DIR'])
if 'KAGGLE_INPUT_PATH' in os.environ and os.path.isdir(os.environ['KAGGLE_INPUT_PATH']):
    candidates.append(os.path.join(os.environ['KAGGLE_INPUT_PATH']))
for root in ('/kaggle/input', '/content', os.path.abspath('.')):
    candidates.append(root)
if os.path.isdir('LGBT'):
    candidates.append(os.path.abspath('LGBT'))

def _first(roots):
    hits = []
    for root in roots:
        hits = sorted(glob.glob(os.path.join(root, '**', 'StereoQueerEval_*_training.tsv'), recursive=True))
        if hits:
            return os.path.dirname(hits[0]), hits
    return None, hits

data_dir, found = _first(candidates)
if not found:
    raise RuntimeError('No StereoQueerEval_*_training.tsv found. Add your Kaggle dataset and re-run.')

os.environ['LGBT_DATA_DIR'] = data_dir
os.environ['LGBT_WEIGHTS_DIR'] = os.path.join(os.getcwd(), 'weights')
print('data dir :', data_dir)
print('files    :', [os.path.basename(p) for p in found])
assert len(found) == 3, f'expect EN+IT+NL, got {len(found)}'

## 2. Sanity check the data

Row counts, label balance, video-grouped split — cheap checks before any GPU time.

In [ ]:
import sys
sys.path.insert(0, os.path.abspath('src'))
import data as data_mod
from config import MAX_LEN

df = data_mod.load_frame(max_len=MAX_LEN)
print('rows:', len(df), '| labels:', data_mod.label_distribution(df))
print('by lang:', df['lang'].value_counts().to_dict())
print('longest prompt words:', int(df['prompt'].str.split().str.len().max()))

tr, va = data_mod.split_by_video(df, 0.1, 42)
print('train/val:', len(tr), '/', len(va))

## 3. Train — `readout_init='random'`

Last 2 encoder blocks unfrozen, `max_len=320`, early stopping on val macro-F1
(`patience=5`). Every epoch logs `train_loss`, `val_loss`, `val_acc`, `val_macro_f1`
and writes them to `weights/..._random_history.csv`. The best checkpoint is saved
per readout variant, so the two runs never overwrite each other.

In [ ]:
!python src/train.py --epochs 10 --batch-size 32 --max-len 320 --unfreeze-layers 2 --readout-init random

## 4. Train — `readout_init='verbalizer'`

Same settings, but the `Linear(768, 2)` starts from the pretrained MLM decoder
rows for `' no'` / `' yes'` instead of random init. Run it and compare the
validation macro-F1 against the previous cell.

In [ ]:
!python src/train.py --epochs 10 --batch-size 32 --max-len 320 --unfreeze-layers 2 --readout-init verbalizer

## 5. Loss and F1 curves

Both `*_history.csv` files have one row per epoch. The verbalizer may still be
running — wait for the cell above to finish first.

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import pandas as pd

def style_ax(ax):
    ax.grid(alpha=0.3, which='both')
    ax.set_xlabel('epoch'); ax.legend()

hist = {
    r: pd.read_csv(os.path.join(os.environ['LGBT_WEIGHTS_DIR'], f'best_mmbert_stereotype_prompt_{r}_history.csv'))
    for r in ('random', 'verbalizer')
    if os.path.exists(os.path.join(os.environ['LGBT_WEIGHTS_DIR'], f'best_mmbert_stereotype_prompt_{r}_history.csv'))
}
print('loaded:', {k: len(v) for k, v in hist.items()})
assert hist, 'no history CSVs found - did the training cells run?'

fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
for r, h in hist.items():
    ax[0].plot(h['epoch'], h['train_loss'], '--', label=f'{r} train')
    ax[0].plot(h['epoch'], h['val_loss'], '-', label=f'{r} val')
    ax[1].plot(h['epoch'], h['val_macro_f1'], '-o', ms=3, label=f'{r}')
ax[0].set_title('loss'); ax[1].set_title('validation macro-F1')
style_ax(ax[0]); style_ax(ax[1])
plt.tight_layout(); plt.show()

best = {r: (h['val_macro_f1'].max(), int(h.loc[h['val_macro_f1'].idxmax(), 'epoch'])) for r, h in hist.items()}
print(best)
winner = max(best, key=best.get)
print('-> best by val macro-F1:', winner, best[winner])

## 6. Confusion matrix on validation (best checkpoint of each variant)

`predict.py` scores the *whole* frame, so it would report in-sample numbers here.
For an honest comparison this cell instead reloads each best checkpoint and
scores the same video-grouped validation split that training used.
`--readout-init` must match training — the flag only selects which checkpoint
file to load, it does not change the saved weights.

In [ ]:
import numpy as np
from sklearn.metrics import confusion_matrix

import torch
from model import load_model, load_model_weights, load_tokenizer
from train import collect, resolve_device

dev = resolve_device('cuda')
tok = load_tokenizer('jhu-clsp/mmbert-base')
va_loader = data_mod.build_inference_loader(va.reset_index(drop=True), tok, 32, MAX_LEN)

preds, labels = {}, {}
for r in hist:
    m = load_model('jhu-clsp/mmbert-base', unfreeze_layers=0, readout_init=r, tokenizer=tok)
    ck = os.path.join(os.environ['LGBT_WEIGHTS_DIR'], f'best_mmbert_stereotype_prompt_{r}.pt')
    load_model_weights(m, ck, device=dev)
    m.to(dev).eval()
    o = collect(m, va_loader, dev)
    preds[r], labels[r] = o['preds'], o['labels']

fig, axes = plt.subplots(1, 2, figsize=(10, 4.2))
for ax, r in zip(axes, hist):
    cm = confusion_matrix(labels[r], preds[r], labels=[0, 1])
    ax.imshow(cm, cmap='Blues')
    ax.set_title(f'{r}')
    ax.set_xticks([0, 1]); ax.set_xticklabels(['no', 'yes'])
    ax.set_yticks([0, 1]); ax.set_yticklabels(['no', 'yes'])
    ax.set_xlabel('predicted'); ax.set_ylabel('true')
    for i in range(2):
        for j in range(2):
            ax.text(j, i, cm[i, j], ha='center', va='center',
                    color='white' if cm[i, j] > cm.max() / 2 else 'black')
plt.suptitle('Confusion matrix on the validation split')
plt.tight_layout(); plt.show()

## 7. Pilot - control vs enriched conditions

Three seeds per condition, same best recipe (warmup 8 -> 32), compared on the
best-epoch validation macro-F1. `--train-extras` is appended **after** the
video-level split, so the validation set stays identical to control and direct
comparison is valid.

Conditions:
- `control` : no extras (locks the baseline - supersedes the old 0.7158/0.6898 ambiguity)
- `tr_nllb` : NLLB back-translations of SQE EN train into IT + NL (run `augment_nllb.py` first)

NOT run (historical): SBIC as direct extras (`sbic_lgbtq`) - measured harmful (0.6593).
The `gen_llm_data.py` / `build_sbic_aux.py` generators were removed by design.

Conditions whose files are missing are skipped by the driver.



In [ ]:
import glob, os, subprocess
import pandas as pd

seeds = [40, 41, 42]
recipe = ["--epochs", "10", "--warmup-batch-size", "8", "--warmup-epochs", "2"]

RES_ROOTS = ['/kaggle/input', os.path.abspath('.'), '/kaggle/working']

def resolve(fname):
    for root in RES_ROOTS:
        hits = sorted(glob.glob(os.path.join(root, '**', fname), recursive=True))
        if hits:
            return hits[0]
    return None

EXTRAS = {
    'control':    None,
    'tr_nllb':    'aug_nllb_EN_to_IT_training.tsv,aug_nllb_EN_to_NL_training.tsv',
    }

results = {}
for name, rel in EXTRAS.items():
    paths = []
    if rel:
        for p in rel.split(','):
            hit = resolve(p)
            if not hit:
                print(f'[skip] {name}: missing -> {p}')
                paths = None
                break
            paths.append(hit)
    if rel and paths is None:
        continue
    for s in seeds:
        out = f'weights/{name}_s{s}.pt'
        cmd = ['python', 'src/train.py', '--languages', 'EN', 'IT', 'NL',
               '--split-seed', str(s), *recipe]
        if paths:
            cmd += ['--train-extras', ','.join(paths)]
        cmd += ['--out', out]
        print('RUN:', ' '.join(cmd))
        subprocess.run(cmd, check=True)
        h = pd.read_csv(out.replace('.pt', '_history.csv'))
        results[(name, s)] = float(h.loc[h['val_macro_f1'].idxmax(), 'val_macro_f1'])

if results:
    rows = []
    for name in {k[0] for k in results}:
        vals = [results[(name, s)] for s in seeds if (name, s) in results]
        row = {'condition': name}
        for s, v in zip(seeds, vals):
            row[f'seed{s}'] = v
        row['mean'] = sum(vals) / len(vals)
        rows.append(row)
    print(pd.DataFrame(rows).to_string(index=False))






## 8. FA zero-shot (cross-lingual)

Score the Farsi development set with a trained checkpoint. The FA prompt now uses
corrected Persian (`کوئیر`, ZWNJ joins, full Persian frame labels).
`predict.py` reports macro-F1 against the `stereotype` column of the dev file.

In [ ]:
import glob, os, subprocess, sys
sys.path.insert(0, os.path.abspath('src'))
import data as data_mod

hits = sorted(glob.glob(os.path.join('/kaggle/input', '**', 'StereoQueerEval_FA_development.tsv'), recursive=True))
if not hits:
    raise RuntimeError('StereoQueerEval_FA_development.tsv not found under /kaggle/input')
FA = hits[0]
print('FA file:', FA)

CKPT = 'weights/control_s40.pt'  # set to your best checkpoint

probe = data_mod.load_frame_from_file(FA, max_len=320)
print(probe['prompt'].iloc[0][-80:])

subprocess.run(['python', 'src/predict.py', '--weights', CKPT, '--input', FA,
                '--label-column', 'stereotype', '--out', 'fa_preds.csv'], check=True)


### Without a notebook

```bash
git clone https://github.com/AmnO-O/Try.git && cd Try
pip install -r requirements.txt
export LGBT_DATA_DIR=/path/to/tsvs            # or put them in ./LGBT/
python src/train.py --epochs 10 --readout-init random
python src/train.py --epochs 10 --readout-init verbalizer
python src/predict.py --readout-init random --eval-only
```

History CSV: `weights/best_..._<readout>_history.csv`. Checkpoints land in
`weights/` (always writable; it is `os.path.join(cwd, 'weights')` here). The
TSVs stay read-only in `/kaggle/input`.